# 노선 ID·정류장 ID 기준 좌표 매핑

`query_route_id + sttnId`로 정류장을 확정한 뒤 국토부 위치정보의 정류장명으로 위도·경도를 연결합니다.

In [ ]:
from pathlib import Path
import re
import pandas as pd

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / 'gtx_a_seoul_bus_outputs' / 'transport_card'
STOP_DIR = DATA_DIR / 'route_stops'
STOPS_FP = Path(r'C:\\Users\\금경훈\\Desktop\\Gachon\\3-2\\UROP\\국토교통부_전국 버스정류장 위치정보_20251031.csv')
DATES = ['20241017', '20251016']

def key(value):
    if pd.isna(value): return ''
    return re.sub(r'\s+', '', str(value).strip())

def find_col(df, candidates):
    for name in candidates:
        if name in df.columns: return name
    raise ValueError(f'필요한 컬럼이 없습니다: {candidates}')

location = pd.read_csv(STOPS_FP, encoding='cp949', dtype=str).fillna('')
name_col = find_col(location, ['정류장명', '정류장 명칭', '버스정류장명'])
lat_col = find_col(location, ['위도', '정류장Y위치값', 'Y좌표'])
lon_col = find_col(location, ['경도', '정류장X위치값', 'X좌표'])
location['_name_key'] = location[name_col].map(key)
location[lat_col] = pd.to_numeric(location[lat_col], errors='coerce')
location[lon_col] = pd.to_numeric(location[lon_col], errors='coerce')
location = location.dropna(subset=[lat_col, lon_col]).drop_duplicates('_name_key')
location_ref = location[['_name_key', lat_col, lon_col]]

for date in DATES:
    raw = pd.read_csv(DATA_DIR / f'gtx_a_transport_card_{date}_raw.csv', dtype=str, encoding='utf-8-sig').fillna('')
    stops = pd.read_csv(STOP_DIR / f'gtx_a_route_stops_{date}.csv', dtype=str, encoding='utf-8-sig').fillna('')
    stops['query_route_id'] = stops['query_route_id'].str.strip()
    stops['sttnId'] = stops['sttnId'].str.strip()
    ref = stops[['query_route_id', 'sttnId', 'sttnNm', 'sttnSeq']].drop_duplicates(['query_route_id', 'sttnId'])

    # 핵심: 수요의 노선 ID + 정류장 ID로 API 정류장명을 확정
    ride_ref = ref.rename(columns={'sttnId':'승차정류장ID', 'sttnNm':'승차정류장명', 'sttnSeq':'승차정류장순서'})
    goff_ref = ref.rename(columns={'sttnId':'하차정류장ID', 'sttnNm':'하차정류장명', 'sttnSeq':'하차정류장순서'})
    result = raw.merge(ride_ref, left_on=['query_route_id','ride_sttn_id'], right_on=['query_route_id','승차정류장ID'], how='left')
    result = result.merge(goff_ref, left_on=['query_route_id','goff_sttn_id'], right_on=['query_route_id','하차정류장ID'], how='left')

    ride_loc = location_ref.rename(columns={'_name_key':'_ride_name_key', lat_col:'승차위도', lon_col:'승차경도'})
    goff_loc = location_ref.rename(columns={'_name_key':'_goff_name_key', lat_col:'하차위도', lon_col:'하차경도'})
    result['_ride_name_key'] = result['승차정류장명'].map(key)
    result['_goff_name_key'] = result['하차정류장명'].map(key)
    result = result.merge(ride_loc, on='_ride_name_key', how='left').merge(goff_loc, on='_goff_name_key', how='left')
    front = ['query_date','query_route_no','query_route_id','승차정류장ID','승차정류장명','승차정류장순서','승차위도','승차경도','하차정류장ID','하차정류장명','하차정류장순서','하차위도','하차경도']
    result = result[[c for c in front if c in result.columns] + [c for c in result.columns if c not in front and not c.startswith('_')]]
    output = DATA_DIR / f'gtx_a_transport_card_{date}_raw_with_coords.csv'
    result.to_csv(output, index=False, encoding='utf-8-sig')
    print(f'{date}: {output.name} 저장 ({len(result):,}건) / 승차 {result["승차위도"].notna().mean():.1%} / 하차 {result["하차위도"].notna().mean():.1%}')